# DT401 Week 5 Section 2b: Unsupervised Model — K-Means Clustering

**Module:** Data Fundamentals
**Week:** 5 of 7
**Section:** 2b of 3
**Estimated Time:** 30 minutes

---

## Learning Objective

Build and interpret a k-means clustering model using real weather data.

By the end of this notebook, you will be able to:
- Explain what an unsupervised clustering model is and when to use it
- Prepare and standardise data for k-means clustering
- Apply the elbow method to select the optimal number of clusters
- Build a k-means model in Python and interpret the cluster centres
- Translate cluster outputs into operational business categories

---

## Context Recap

In Section 2a, you built a supervised regression model to predict apparent temperature — a specific numerical output. This notebook covers the second major model type: **unsupervised clustering**, which discovers natural groupings in data without a dependent variable to predict.

We continue with the same London weather dataset and the same business context — an outdoor events company — but now ask a different question: **what types of weather does London typically experience?**

**Weather data from Open-Meteo API (https://open-meteo.com) — London, UK (51.49°N, 0.16°W), Jan 2023–Jan 2026**

---

## How to Use This Notebook

This notebook contains a mix of instructional text, Python code cells, and interactive exercises. Follow the instructions in each section, run the code cells to see outputs, and complete the exercises to reinforce your learning.

You will find several types of boxes throughout the notebook to guide your learning:

<div class="alert alert-block alert-warning">
<b>Reminder boxes:</b> <br>
These boxes contain important reminders about how this section relates to the overall module and your learning journey.
</div>

<div class="alert alert-block alert-info">
<b>Reflection boxes:</b> <br>
These have pointers as to what we should be looking for or comments on how to approach the topic discussed.
</div>

<div class="alert alert-block alert-success">
<b>Task boxes:</b> <br>
These have questions or tasks you should complete before progressing.

These may include hidden hints or solutions.
<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Hints help guide your thinking without giving away the full answer.

</details>

</div>

---

## Part 1: What Is an Unsupervised Clustering Model?

### Visual Overview

```mermaid
graph TD
    A[Weather Observations<br/>No predefined labels] --> B[K-Means Clustering]
    B --> C[Cluster 0:<br/>Rainy]
    B --> D[Cluster 1:<br/>Cold & Overcast]
    B --> E[Cluster 2:<br/>Clear & Dry]
    B --> F[Cluster 3:<br/>Mild & Breezy]

    style A fill:#e1f5ff
    style B fill:#fff4e1
    style C fill:#ffe1e1
    style D fill:#e1e1ff
    style E fill:#e1ffe1
    style F fill:#fff4cc
```

**What this diagram shows:**
K-means clustering groups observations based on similarity without any predefined labels. The algorithm discovers the groups — we interpret and name them afterwards.

**Why this matters in your role:**
Clustering reveals hidden structure useful for segmentation. Use it when you want to understand natural groupings in your data — customer types, risk profiles, operational categories — without having predefined answers.

---

### Business Scenario

The outdoor events company wants to move beyond individual forecasts and build an operational classification system: **what type of weather day is it?** Different weather types require different staffing, equipment, and contingency plans — but the company currently decides these case-by-case.

Clustering will allow them to discover natural weather categories from three years of data, then assign future days to those categories automatically to trigger the appropriate operational response.

---

### Approach

We use the same Open-Meteo dataset, but this time we do not predict a specific value. Instead, we ask: **what natural categories of weather does London experience?** K-means will discover the answer from the data itself.

We use four variables that capture distinct dimensions of weather conditions:

| Variable | Column | What it captures |
| :--- | :--- | :--- |
| Temperature (°C) | `temperature_2m` | How warm or cold conditions feel |
| Cloud cover (%) | `cloud_cover` | How bright or grey the day is |
| Wind speed (km/h) | `wind_speed_10m` | How exposed attendees and structures are |
| Precipitation (mm) | `precipitation` | Whether it is raining |

---

### How K-Means Clustering Works

K-means clustering groups observations into **k** clusters based on similarity. Unlike regression, there is no dependent variable — the algorithm discovers natural groupings by minimising the total distance between each data point and its nearest cluster centre.

**The algorithm works iteratively:**
1. Place k cluster centres randomly in the data space
2. Assign each observation to its nearest centre (measured by Euclidean distance)
3. Move each centre to the mean of the observations assigned to it
4. Repeat steps 2–3 until assignments stop changing

The result minimises the **inertia** — the total squared distance between all observations and their assigned cluster centres. This is the same Sum Squared Error (SSE) principle used in linear regression.

---

### Requirements and Conditions

**Requirements:** One or more **independent variables** — all numerical, with at least some continuous values.

**Conditions for good clustering:**
- Variables should be **independent** of each other. Highly correlated variables carry the same information — only one is needed.
- Variables must be on a **similar scale**. A variable ranging 0–100 (cloud cover) would dominate a variable ranging 0–3 (precipitation) purely because of its size.

Because our four variables have very different scales, all must be **standardised** before clustering — you will do this in Step 2.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Load weather dataset
weather_data = pd.read_csv('../open-meteo-51.49N0.16W23m.csv', skiprows=3, low_memory=False)
weather_data.columns = [c.split(' ')[0].strip() for c in weather_data.columns]
weather_data = weather_data.rename(columns={'time': 'date'})

# Keep only hourly rows (YYYY-MM-DDTHH:MM format)
# The CSV contains a second daily-data section appended after the hourly data
weather_data = weather_data[weather_data['date'].str.match(r'^\d{4}-\d{2}-\d{2}T', na=False)].reset_index(drop=True)

# Convert all measurement columns to numeric
for col in weather_data.columns[1:]:
    weather_data[col] = pd.to_numeric(weather_data[col], errors='coerce')
weather_data['wind_speed_10m'] = weather_data['wind_speed_10m'] * 1.60934

print(f"Dataset loaded: {len(weather_data):,} hourly observations")
print(f"Date range: {weather_data['date'].min()} to {weather_data['date'].max()}")
weather_data[['date', 'temperature_2m', 'cloud_cover', 'wind_speed_10m', 'precipitation']].head()

---

## Part 2: Building the Clustering Model

### Step 1 — Select and Prepare Variables

In [ ]:
# Select four variables that capture distinct aspects of weather conditions
cluster_data = weather_data[['temperature_2m', 'cloud_cover', 'wind_speed_10m', 'precipitation']].copy()
cluster_data = cluster_data.dropna()

# Sample 10,000 observations for computational efficiency
cluster_sample = cluster_data.sample(n=10000, random_state=42)

print(f"Clustering dataset: {len(cluster_sample):,} observations")
print()
print("Summary statistics:")
cluster_sample.describe().round(2)

<div class="alert alert-block alert-warning">
<b>Reminder — why we need to standardise:</b> <br>
K-means measures distance between observations. A variable ranging 0–100 (cloud cover) will dominate a variable ranging 0–3 (precipitation) in that distance calculation — not because it is more important, but because it is larger. Standardising each variable to mean=0, std=1 gives every variable equal weight.
</div>

### Step 2 — Standardise the Variables

In [ ]:
# Standardise all variables (mean=0, std=1)
scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_sample)

# Confirm standardisation
scaled_df = pd.DataFrame(cluster_scaled, columns=cluster_sample.columns)
print("After standardisation — each variable now has mean=0 and std=1:")
scaled_df.describe().round(2)

### Step 3 — Find the Optimal Number of Clusters (Elbow Method)

As the number of clusters increases, each cluster becomes smaller and the total inertia decreases. But the improvement gets smaller with each additional cluster. We look for the **elbow** — the point where adding one more cluster gives diminishing returns.

The plot below shows inertia for k=2 through k=8. The curve drops sharply up to k=4, then flattens — meaning the 4th cluster adds meaningful structure, but the 5th adds comparatively little:

![Elbow method: selecting the number of weather type clusters](../images/cluster_elbow.png)

Run the code below to generate this plot from your data and confirm the elbow at k=4.

In [ ]:
# Test k=2 through k=8 — record inertia for each
inertias = []
k_range = range(2, 9)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(cluster_scaled)
    inertias.append(kmeans.inertia_)
    print(f"k={k}: inertia = {kmeans.inertia_:.1f}")

# Plot elbow
plt.figure(figsize=(9, 5))
plt.plot(list(k_range), inertias, marker='o', color='steelblue', lw=2, ms=8)
plt.axvline(4, color='red', linestyle='--', lw=1.5, label='Elbow at k=4')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (Sum of Squared Distances)')
plt.title('Elbow Method: Finding Optimal Number of Weather Clusters')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Inertia drops sharply from k=2 to k=4, then flattens.")
print("k=4 balances statistical fit with operational usability.")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We fitted k-means models for k=2 through k=8 and recorded the <b>inertia</b> (total SSE) for each
- Inertia always decreases as k increases — more clusters means each cluster is smaller and tighter
- The <b>elbow</b> is the point where adding another cluster gives diminishing returns
- At k=4, the improvement from adding one more cluster drops noticeably — this is our optimal k

<b>Key lesson:</b> The elbow method is a judgement call, not a precise formula. Balance the statistical improvement against the practical question: are four categories more useful than three?
</div>

### Step 4 — Build the Final k=4 Model

In [ ]:
# Fit the final model with k=4
final_kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_sample = cluster_sample.copy()
cluster_sample['cluster'] = final_kmeans.fit_predict(cluster_scaled)

# Reverse the standardisation to get interpretable cluster centres
centres = pd.DataFrame(
    scaler.inverse_transform(final_kmeans.cluster_centers_),
    columns=['temperature_2m', 'cloud_cover', 'wind_speed_10m', 'precipitation']
).round(1)

sizes = cluster_sample['cluster'].value_counts().sort_index()
centres['observations'] = sizes.values
centres['share_%'] = (sizes.values / len(cluster_sample) * 100).round(1)

# Dynamically assign weather labels based on cluster characteristics.
# k-means cluster numbering is arbitrary — labels are determined by
# which cluster wins each characteristic, in priority order.
assigned = {}

# 1. Highest precipitation → Rainy
rainy_idx = int(centres['precipitation'].idxmax())
assigned[rainy_idx] = 'Rainy'

# 2. Lowest cloud cover (among remaining) → Clear & dry
remaining = [i for i in range(4) if i not in assigned]
clear_idx = int(centres.loc[remaining, 'cloud_cover'].idxmin())
assigned[clear_idx] = 'Clear & dry'

# 3. Lowest temperature (among remaining) → Cold & overcast
remaining = [i for i in range(4) if i not in assigned]
cold_idx = int(centres.loc[remaining, 'temperature_2m'].idxmin())
assigned[cold_idx] = 'Cold & overcast'

# 4. Remaining cluster → Mild & breezy
remaining = [i for i in range(4) if i not in assigned]
assigned[remaining[0]] = 'Mild & breezy'

weather_labels = assigned

# Display labelled cluster centres
display_centres = centres.copy()
display_centres.index = [f"Cluster {i}: {weather_labels[i]}" for i in display_centres.index]
print("=== CLUSTER CENTRES (unscaled) ===")
print(display_centres.to_string())

### Interpreting the Cluster Centres

The model assigns each of the 10,000 observations to one of four clusters. The cluster centres — expressed in original, unscaled units — reveal what each cluster represents:

| Cluster | Weather type | Temperature (°C) | Cloud cover (%) | Wind speed (km/h) | Precipitation (mm) | Share |
|:-------:|:------------|:----------------:|:---------------:|:-----------------:|:-----------------:|:-----:|
| 0 | Rainy | 12.3 | 97.9 | 17.1 | 2.0 | 2.2% |
| 1 | Cold & overcast | 9.8 | 95.3 | 9.5 | 0.0 | 37.2% |
| 2 | Clear & dry | 11.6 | 13.8 | 11.1 | 0.0 | 31.5% |
| 3 | Mild & breezy | 14.7 | 86.6 | 21.5 | 0.1 | 29.1% |

London's weather is dominated by two patterns: **cold overcast hours** (37%) and **clear dry conditions** (32%), with mild breezy conditions adding another 29%. Genuinely rainy hours — with measurable precipitation — represent only **2.2%** of observations.

The cluster scatter below shows these categories clearly when temperature and cloud cover are plotted together:

![Weather clusters: temperature vs cloud cover](../images/cluster_scatter.png)

The primary drivers separating clusters are **cloud cover** and **temperature**. Wind speed adds further distinction for the mild & breezy and rainy clusters. Run the code below to generate this scatter from your data:

### Step 5 — Visualise the Clusters

In [ ]:
# Scatter plot: temperature vs cloud cover, coloured by cluster
colours = ['#e05c24', '#3a7dc9', '#2ca02c', '#9467bd']

plt.figure(figsize=(10, 6))
for c in range(4):
    sub = cluster_sample[cluster_sample['cluster'] == c]
    plt.scatter(sub['temperature_2m'], sub['cloud_cover'],
                alpha=0.3, s=6, color=colours[c],
                label=f"Cluster {c}: {weather_labels[c]}")

plt.xlabel('Temperature (°C)')
plt.ylabel('Cloud Cover (%)')
plt.title('Weather Type Clusters — London 2023–2026 (Temperature vs Cloud Cover)')
plt.legend(markerscale=4, fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We built a k=4 model and assigned each of the 10,000 observations to its nearest cluster
- We reversed the standardisation to express cluster centres in original, interpretable units
- We visualised clusters using temperature and cloud cover — the two variables that most clearly separate the groups

<b>Interpreting the four weather types:</b>
<ul>
<li><b>Rainy (2.2%):</b> High cloud, measurable precipitation, moderate wind — stormy conditions. Rare but operationally critical.</li>
<li><b>Cold & overcast (37.2%):</b> Low temperature, near-total cloud cover, calm winds. The dominant London pattern — grey, still winter days.</li>
<li><b>Clear & dry (31.5%):</b> Low cloud cover, no rain, moderate wind. Bright conditions — best for outdoor events.</li>
<li><b>Mild & breezy (29.1%):</b> Warmest cluster, overcast, highest wind speeds. Comfortable but requires wind management for temporary structures.</li>
</ul>

<b>Key lesson:</b> The algorithm finds the clusters; you determine what they mean and how to use them. Domain knowledge is essential for interpretation.
</div>

### Operational Framework

For the outdoor events company, these four weather types provide a clear, actionable operational framework:

| Weather type | Share | Operational response |
| :--- | :---: | :--- |
| Clear & dry | 31.5% | No contingency required — best conditions for outdoor events |
| Cold & overcast | 37.2% | Provide heating; warm clothing advisory for attendees |
| Mild & breezy | 29.1% | Wind anchoring required for temporary structures and signage |
| Rainy | 2.2% | Full rain contingency: cover deployment, equipment protection, possible rescheduling |

Rather than making these decisions case-by-case for each event, the model allows the company to define **standard operating procedures** for each weather type — applied automatically when a forecast day is assigned to a cluster.

This is the key operational value of clustering: it converts continuous, multidimensional weather data into a small set of **named categories** that non-technical staff can act on directly.

---

## Part 2: Modified Task — Experiment with k=3

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Rebuild the clustering model with <b>k=3</b> clusters. Examine the cluster centres and scatter plot.

Which value of k — 3 or 4 — produces more operationally useful weather categories? Justify your answer.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Steps:
1. Create KMeans(n_clusters=3, random_state=42, n_init=10)
2. Fit on cluster_scaled
3. Add a 'cluster_3' column to cluster_sample
4. Inverse-transform the centres using scaler.inverse_transform()
5. Visualise and compare cluster characteristics to k=4

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_sample['cluster_3'] = km3.fit_predict(cluster_scaled)

centres_3 = pd.DataFrame(
    scaler.inverse_transform(km3.cluster_centers_),
    columns=['temperature_2m', 'cloud_cover', 'wind_speed_10m', 'precipitation']
).round(1)
sizes_3 = cluster_sample['cluster_3'].value_counts().sort_index()
centres_3['share_%'] = (sizes_3.values / len(cluster_sample) * 100).round(1)
print(centres_3)

plt.figure(figsize=(9, 5))
for c in range(3):
    sub = cluster_sample[cluster_sample['cluster_3'] == c]
    plt.scatter(sub['temperature_2m'], sub['cloud_cover'], alpha=0.3, s=6, label=f'Cluster {c}')
plt.xlabel('Temperature (°C)')
plt.ylabel('Cloud Cover (%)')
plt.title('k=3 Weather Clusters')
plt.legend(markerscale=4)
plt.grid(True, alpha=0.3)
plt.show()
```

k=3 merges the 'Rainy' and 'Mild & breezy' clusters — losing the distinction between stormy and breezy conditions. For event planning, this distinction matters: k=4 is more actionable.

</details>

</div>

In [ ]:
# Your code here: build a k=3 clustering model and compare to k=4

# 1. Create and fit a KMeans model with n_clusters=3


# 2. Assign clusters to cluster_sample as 'cluster_3'


# 3. Reverse-transform cluster centres and display


# 4. Create a scatter plot coloured by cluster_3 (optional but recommended)


print("k=3 results:")

**Expected output:** Cluster centres for k=3 and a scatter plot

**Success check:** You can compare k=3 and k=4 and justify which number of clusters is more useful for operational decision-making at the events company.

---

## Part 3: Independent Task — Model Selection for Your Workplace

### Your Challenge

You work for a local council managing community services. The council has data on:
- Service requests (type, location, priority, resolution time)
- Resource allocation (staff hours, equipment, costs)
- Citizen satisfaction scores
- Geographic data (neighbourhood demographics, service accessibility)

**Your manager asks: "Can data analysis help us improve service delivery?"**

Design **three** different analytical approaches:

1. **Regression model:** What continuous variable could you predict? What predictors would you use? What business decision does this support?
2. **Classification model:** What categories could you predict? What features would indicate membership? What actions follow from each prediction?
3. **Clustering model:** What natural groupings might exist? What variables define similarity? How would the clusters be used operationally?

**For each model, define:**
- The business problem it addresses
- The dependent variable (regression/classification) or clustering variables
- The independent variables / features
- The actionable insights the model provides
- How you would measure success

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

**Regression:** Predict service resolution time (hours) from request type, priority, neighbourhood, time of day.
**Classification:** Predict priority level (Low / Medium / High / Urgent) from request description, location, historical patterns.
**Clustering:** Group neighbourhoods by service volume, response time, satisfaction scores.

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
model_designs = {
    'regression': {
        'business_problem': 'Predict service resolution time to set accurate citizen expectations',
        'dependent_variable': 'Resolution time (hours)',
        'independent_variables': ['Request type', 'Priority', 'Neighbourhood', 'Time of day', 'Staff on duty'],
        'actionable_insights': 'Show citizens realistic timeframes at submission; flag high-delay scenarios for resource reallocation',
        'success_metric': 'RMSE < 4 hours'
    },
    'classification': {
        'business_problem': 'Auto-prioritise service requests so urgent issues are handled first',
        'categories': ['Low', 'Medium', 'High', 'Urgent'],
        'features': ['Request description (text)', 'Location risk', 'Historical patterns', 'Time since last service'],
        'actionable_insights': 'Route Urgent requests to senior staff automatically; batch Low requests for scheduled maintenance',
        'success_metric': '85% classification accuracy; all Urgent predictions reviewed by senior staff'
    },
    'clustering': {
        'business_problem': 'Identify neighbourhoods with similar service needs for targeted resource allocation',
        'clustering_variables': ['Service volume', 'Average response time', 'Satisfaction scores', 'Demographics'],
        'actionable_insights': 'High-need clusters get proactive outreach and additional resources; low-need clusters optimise for efficiency',
        'success_metric': 'Clusters show distinct profiles; satisfaction improves 20% in high-need clusters after reallocation'
    }
}
```

</details>

In [ ]:
# Your model designs — complete this for your portfolio
model_designs = {
    'regression': {
        'business_problem': '',
        'dependent_variable': '',
        'independent_variables': [],
        'actionable_insights': '',
        'success_metric': ''
    },
    'classification': {
        'business_problem': '',
        'categories': [],
        'features': [],
        'actionable_insights': '',
        'success_metric': ''
    },
    'clustering': {
        'business_problem': '',
        'clustering_variables': [],
        'actionable_insights': '',
        'success_metric': ''
    }
}

for model_type, design in model_designs.items():
    print(f"\n=== {model_type.upper()} MODEL ===")
    for key, value in design.items():
        print(f"  {key}: {value}")

**Success criteria:**
- All three model types designed with clear business problems
- Appropriate variables/features identified for each approach
- Actionable insights clearly stated — not just technical outputs
- Success metrics are specific and measurable

**Portfolio note:** This model selection task maps to MLO 3. Include it in your Module Challenge Part A portfolio.

---

## Reflection

Reflect on the following question and record your thoughts in the cell below. Ground your answer in an example from your workplace, using **approximately 150 words** in total.

Your answer here forms part of your Module Challenge Part A portfolio and is assessed against **MLO3:** *Select and apply appropriate data-gathering methods for a given data analysis scenario*.

Where you draw on external sources, cite them in Harvard style. This does not count towards your word count.

Once you commit your notebook, your Coach will be able to provide feedback on your response. You can develop your answer further before your final submission.

<div class="alert alert-block alert-info">

1. Would stakeholders in your role care more about a clustering model's accuracy or the interpretability of the segments it produces? Why? How does this affect your choice of model?

</div>

*Write your responses here.*

1. 

References (if applicable):

---

## Summary

**You have completed Week 5 Section 2b.** You can now:

- Explained what an unsupervised clustering model is, distinguishing it from regression: k-means groups observations by similarity with no dependent variable to predict
- Standardised four weather variables (temperature, cloud cover, wind speed, precipitation) with `StandardScaler`, applied the elbow method across k=2 to k=8, and built a final `KMeans(n_clusters=4)` model, naming the resulting clusters Rainy, Cold & overcast, Clear & dry, and Mild & breezy
- Rebuilt the model with k=3 and compared its cluster centres to the k=4 solution to judge which produces more operationally useful weather categories
- Designed three analytical approaches (regression, classification, clustering) for a local council service-delivery problem, defining variables, features, and success metrics for each

**Key takeaways:**

- Standardise variables before clustering: without it, a variable with a larger numeric range (cloud cover, 0-100) dominates the distance calculation over one with a smaller range (precipitation, 0-3), regardless of its real importance
- The elbow method is a judgement call, not a precise formula: it weighs the statistical drop in inertia against how operationally useful the resulting number of categories actually is

**What comes next:**

- **Section 3: Testing a Hypothesis** replaces this kind of visual judgement call with formal statistical tests, giving you a rigorous way to decide whether a pattern in your data is genuine rather than a matter of interpretation.

**Action items:**

1. Save this notebook (Ctrl+S or Cmd+S)
2. Commit and push your work to your GitHub repository with the commit message `"Complete Week 5 Section 2b K-Means Clustering"`
3. Make sure the repository shows your latest commit and that all files are up to date
4. Share a link to your repository with your Coach in the DT401.5 component on Aptem
5. Open `DT401_Week5_Section3.ipynb` when ready to continue